# EDA — Données environnementales (météo, sol, relief)

Exploration des données téléchargées, et réflexion sur la **fusion** vers le
dataset d'entraînement.

Sources : Open-Meteo (météo, 46 points × 9 783 jours), SoilGrids (sol, 46 points),
AWS terrain-tiles (DEM, 40 tuiles).


In [ ]:
import math
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
import matplotlib.pyplot as plt
import seaborn as sns

from champiquest.data.gbif import RAW_DIR
from champiquest.data.openmeteo import OUT_DIR as WEATHER_DIR
from champiquest.data.environnement import DEM_DIR, SOIL_DIR

GRID = 0.25  # grille des points météo/sol

weather_files = sorted(WEATHER_DIR.glob("*.csv"))
weather = {f.stem: pd.read_csv(f, index_col=0, parse_dates=True) for f in weather_files}
soil = pd.read_csv(SOIL_DIR / "soil_points.csv")

print(f"{len(weather)} séries météo | sol : {soil.shape[0]} points × {soil.shape[1]} colonnes")
print("Variables météo :", list(weather[weather_files[0].stem].columns))


In [ ]:
# Corrélation entre les variables quotidiennes (sur un point, représentatif)
corr = weather[weather_files[0].stem].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Corrélation des 5 variables météo quotidiennes")
plt.show()

# Climatologie mensuelle (moyenne des 46 points)
allw = pd.concat(weather.values())
allw["month"] = allw.index.month
monthly = allw.groupby("month")[["temperature_2m_mean", "precipitation_sum"]].mean()
monthly.plot(subplots=True, figsize=(8, 5), title="Climatologie mensuelle (moyenne des 46 points)")
plt.show()


In [ ]:
# Collinéarité : la texture somme à 100 % ?
tex_sum = soil["sand_0-5cm"] + soil["silt_0-5cm"] + soil["clay_0-5cm"]
print("somme sand+silt+clay :", tex_sum.round(1).unique())

props = ["phh2o_0-5cm", "sand_0-5cm", "silt_0-5cm", "clay_0-5cm", "soc_0-5cm"]
sns.heatmap(soil[props].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Corrélation des propriétés de sol")
plt.show()

soil[props].hist(bins=15, figsize=(12, 6))
plt.suptitle("Distributions des propriétés de sol")
plt.show()


In [ ]:
to_merc = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

def elevation(lat, lon, zoom=10):
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    y = int((1 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2 * n)
    mx, my = to_merc.transform(lon, lat)
    with rasterio.open(DEM_DIR / f"{zoom}_{x}_{y}.tif") as src:
        r, c = src.index(mx, my)
        return float(src.read(1)[r, c])

# Altitude aux 46 points de grille vs aux observations exactes
grid_alt = [elevation(r["lat"], r["lon"]) for _, r in soil.iterrows()]
print(f"Altitude des 46 points de grille : min {min(grid_alt):.0f} m | médiane {np.median(grid_alt):.0f} m | max {max(grid_alt):.0f} m")

occ = pd.concat([pd.read_csv(f, usecols=["decimalLatitude", "decimalLongitude"]) for f in RAW_DIR.glob("*.csv")], ignore_index=True)
sample = occ.sample(300, random_state=0)
obs_alt = [elevation(r["decimalLatitude"], r["decimalLongitude"]) for _, r in sample.iterrows()]
print(f"Altitude de 300 observations exactes : min {min(obs_alt):.0f} m | médiane {np.median(obs_alt):.0f} m | max {max(obs_alt):.0f} m")


In [ ]:
# Combien d'observations tombent sur chaque point de grille ?
occ = pd.concat([pd.read_csv(f, usecols=["decimalLatitude", "decimalLongitude"]) for f in RAW_DIR.glob("*.csv")], ignore_index=True)
occ["glat"] = (occ["decimalLatitude"] / GRID).round() * GRID
occ["glon"] = (occ["decimalLongitude"] / GRID).round() * GRID
counts = occ.groupby(["glat", "glon"]).size()
counts.hist(bins=30)
plt.title("Nombre d'observations par point de grille")
plt.show()
print(counts.describe().round(1).to_string())


## 🔀 Stratégie de fusion (vers le dataset d'entraînement)

**Schéma cible : 1 ligne = 1 observation** (+ pseudo-absences, générées plus tard).

| Colonne | Source | Commentaire |
|---|---|---|
| `target` | 1 = présence, 0 = pseudo-absence | |
| `species` | GBIF | identifiant de l'observation |
| features météo (J-14/J-60, choc thermique, bilan hydrique…) | Open-Meteo via `features/weather.py` | calculées depuis la série du **point de grille** (0.25°) |
| `ph`, `sand`, `clay`, `soc` | SoilGrids | au **point de grille** — `silt` exclu (collinéaire) |
| `elevation`, `slope`, `aspect` | DEM | échantillonnées aux **coordonnées exactes** (l'altitude varie fortement dans une maille) |
| `forest_type` | CLC | aux coordonnées exactes (à venir) |

### Découvertes à retenir
- **Météo** : précipitations et température quasi indépendantes (bonnes features) ; ET0 très corrélé à la température.
- **Sol** : `sand + silt + clay = 100 %` → collinéarité parfaite → **garder 2 des 3** (ex. sand + clay) + pH + soc.
- **Altitude** : les 46 points de grille plafonnent à ~1 000 m, mais les observations exactes montent à ~1 400 m → **échantillonner l'altitude aux coordonnées exactes**, pas à la grille.
- **Clustering** : 1 point de grille concentre 1 259 observations (médiane 20) → biais d'échantillonnage à surveiller pour les pseudo-absences.
